# Агентная модель SIR: многокритериальная оптимизация

Ищем сочетания параметров, при которых одновременно малы пиковая
заболеваемость и доля умерших. Используется эволюционный алгоритм
Borg MOEA из пакета BlackBoxOptim.

## Активация проекта и загрузка пакетов

In [1]:
using DrWatson
@quickactivate "project"

using BlackBoxOptim, Random, Statistics

include(srcdir("sir_model.jl"))

total_count (generic function with 1 method)

## Целевая функция

Вектор `x` содержит три параметра: `x[1]` --- заразность $\beta$,
`x[2]` --- время выявления, `x[3]` --- вероятность смерти.
Для каждого набора выполняем пять прогонов с разными зёрнами и
возвращаем средний пик заболеваемости и среднюю долю умерших.

In [2]:
function cost_multi(x)
    infected_frac(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
    dead_count(model) = 3000 - nagents(model)

    replicates = 5
    peak_vals = Float64[]
    dead_vals = Int[]

    for rep = 1:replicates
        model = initialize_sir(;
            Ns = [1000, 1000, 1000],
            β_und = fill(x[1], 3),
            β_det = fill(x[1] / 10, 3),
            infection_period = 14,
            detection_time = round(Int, x[2]),
            death_rate = x[3],
            reinfection_probability = 0.1,
            Is = [0, 0, 1],
            seed = 42 + rep,
            n_steps = 100,
        )

        peak_infected = 0.0
        for step = 1:100
            Agents.step!(model, 1)
            frac = infected_frac(model)
            if frac > peak_infected
                peak_infected = frac
            end
        end

        push!(peak_vals, peak_infected)
        push!(dead_vals, dead_count(model))
    end

    return (mean(peak_vals), mean(dead_vals) / 3000)
end

cost_multi (generic function with 1 method)

## Запуск оптимизации

Границы поиска: $\beta$ от 0.1 до 1.0, время выявления от 3 до 14
дней, вероятность смерти от 0.01 до 0.1. Оптимизация идёт две минуты.

In [3]:
result = bboptimize(
    cost_multi,
    Method = :borg_moea,
    FitnessScheme = ParetoFitnessScheme{2}(is_minimizing = true),
    SearchRange = [
        (0.1, 1.0),
        (3.0, 14.0),
        (0.01, 0.1),
    ],
    NumDimensions = 3,
    MaxTime = 120,
    TraceMode = :compact,
)

Starting optimization with optimizer BlackBoxOptim.BorgMOEA{EpsBoxDominanceFitnessScheme{2, Float64, true, typeof(sum)}, BlackBoxOptim.ProblemEvaluator{Tuple{Float64, Float64}, IndexedTupleFitness{2, Float64}, EpsBoxArchive{2, Float64, EpsBoxDominanceFitnessScheme{2, Float64, true, typeof(sum)}}, FunctionBasedProblem{typeof(cost_multi), ParetoFitnessScheme{2, Float64, true, typeof(sum)}, ContinuousRectSearchSpace, Nothing}}, FitPopulation{IndexedTupleFitness{2, Float64}}, FixedGeneticOperatorsMixture, RandomBound{ContinuousRectSearchSpace}}
0.00 secs, 0 evals, 0 steps
pop.size=50 arch.size=0 n.restarts=0

Optimization stopped after 1 steps and 388.81 seconds
Termination reason: Max time (120.0 s) reached
Steps per second = 0.00
Function evals per second = 0.13
Improvements/step = NaN
Total function evaluations = 51


Best candidate found: [0.141328, 3.7445, 0.0511145]

Fitness: (0.00040, 0.00000) agg=0.00040



BlackBoxOptim.OptimizationResults("borg_moea", "Max time (120.0 s) reached", 1, 1.791347673798777e9, 388.80702900886536, ParamsDictChain[ParamsDictChain[Dict{Symbol, Any}(:NumDimensions => 3, :MaxTime => 120.0, :SearchRange => [(0.1, 1.0), (3.0, 14.0), (0.01, 0.1)], :TraceMode => :compact, :Method => :borg_moea, :FitnessScheme => ParetoFitnessScheme{2, Float64, true, typeof(sum)}(sum), :MaxFuncEvals => 0, :MaxSteps => 0),Dict{Symbol, Any}()],Dict{Symbol, Any}(:CallbackInterval => -1.0, :TargetFitness => nothing, :TraceMode => :compact, :FitnessScheme => ScalarFitnessScheme{true}(), :MinDeltaFitnessTolerance => 1.0e-50, :NumDimensions => :NotSpecified, :FitnessTolerance => 1.0e-8, :TraceInterval => 0.5, :MaxStepsWithoutProgress => 10000, :MaxSteps => 10000…)], 51, EpsBoxDominanceFitnessScheme{2, Float64, true, typeof(sum)}([0.1, 0.1], sum), BlackBoxOptim.EpsBoxArchiveOutput{2, Float64, EpsBoxDominanceFitnessScheme{2, Float64, true, typeof(sum)}}((0.0004, 0.0), [0.14132822258996208, 3.74

## Результаты

In [4]:
best = best_candidate(result)
fitness = best_fitness(result)

println("Оптимальные параметры:")
println("β_und = $(best[1])")
println("Время выявления = $(round(Int, best[2])) дней")
println("Смертность = $(best[3])")
println("Достигнутые показатели:")
println("Пик заболеваемости: $(fitness[1])")
println("Доля умерших: $(fitness[2])")

Оптимальные параметры:
β_und = 0.14132822258996208
Время выявления = 4 дней
Смертность = 0.05111449932793503
Достигнутые показатели:
Пик заболеваемости: 0.0004
Доля умерших: 0.0


## Сохранение результатов

In [5]:
wsave(
    datadir("optimization_result.jld2"),
    Dict("best" => collect(best), "fitness" => [fitness[1], fitness[2]]),
)